# Continue a saved model and measure learning versus protection

**Target:** load the correct saved model, teach one new task, and see when earlier scores fall.

The training objective is `new-task loss + beta ? protection penalty`. This notebook records both terms separately. Once per pass through the images, it also measures the size and direction of their gradients on the shared generator. Gradients describe how a loss pushes the model's numbers.

First use a **CPU session** to choose and inspect the saved model. Then use a fresh **A100 GPU session** to train. The default runs one beta value, `0.01`. There is no training in SELECT mode.

If a model after learning task 5 exists, select it to diagnose learning 17. Otherwise select a model after learning 9 to diagnose learning 5. A model that has already learned 17 cannot be used to repeat its earlier lesson.


## 1. Connect Drive and load the code

In the first session, choose CPU in Colab. Run this cell and allow access to Drive. It loads a fixed code version. Run the same cell again when you later open a fresh GPU session.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
from google.colab import drive

drive.mount("/content/drive")
REPO = Path("/content/uncle-learning-loss")
CODE_VERSION = "6a59648a5512724f94917ed9b97a12807c68809b"
assert not any(n == "uncle" or n.startswith("uncle.") for n in sys.modules), "Start a fresh runtime before loading this code again."
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/sumitasthana/CARK.git", str(REPO)], check=True)
else:
    assert (REPO / ".git").is_dir(), "Use a fresh session."
    remote = subprocess.check_output(["git", "-C", str(REPO), "remote", "get-url", "origin"], text=True).strip().removesuffix(".git")
    assert remote == "https://github.com/sumitasthana/CARK", "This folder belongs to another project."
    assert not subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain"], text=True).strip(), "Save local code changes and use a fresh session."
subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", CODE_VERSION], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", "-e", str(REPO)], check=True)
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print("Code version:", CODE_VERSION)


## 2. Choose the mode and find saved models

Leave `MODE = "SELECT"` for the CPU session. The cell prints numbered saved-model paths. Files under your Hyperfan experiment folders are the likely candidates.

Keep the same experiment name when moving to GPU. For a different source model, next task, or retry after an interrupted lesson, choose a new experiment name.

In the GPU session, change MODE to `"RUN"`. Leave BETAS at `(0.01,)` for the first diagnostic. The notebook will read the choice you saved during the CPU session.


In [ ]:
import json
import math
import torch
from uncle.learning_diagnostic import inspect_source, file_hash, run_learning_diagnostic, summarize_diagnostic

MODE = "SELECT"  # SELECT on CPU first; RUN on a fresh A100 afterwards.
EXPERIMENT_ID = "20261006_learning_loss_01"
BETAS = (0.01,)  # One diagnostic first. Optional later search: (0.001, 0.01, 0.1, 1.0).
ROOT = Path("/content/drive/MyDrive/uncle")
OUTPUT_ROOT = ROOT / "learning_loss_diagnostic" / EXPERIMENT_ID
DATA = Path("/content/data/tiny-imagenet-200")
SELECTION_PATH = OUTPUT_ROOT / "selection.json"
assert MODE in ("SELECT", "RUN")
assert BETAS and len(set(BETAS)) == len(BETAS)
assert all(math.isfinite(b) and b >= 0 for b in BETAS)
if MODE == "SELECT":
    candidates = sorted(ROOT.rglob("*.pt"))
    assert candidates, f"No saved models were found under {ROOT}. Check the Drive folder."
    for number, path in enumerate(candidates, 1):
        print(f"{number}: {path}")
    print("Choose a file number in the next cell. No model has been trained.")
else:
    assert SELECTION_PATH.is_file(), "Choose and save a source model in SELECT mode first."
    selection = json.loads(SELECTION_PATH.read_text())
    assert selection["code_version"] == CODE_VERSION, "Use the code version saved with this selection."
    assert torch.cuda.is_available(), "Choose an A100 GPU runtime."
    assert "A100" in torch.cuda.get_device_name(0), "Choose A100 for this ResNet50 diagnostic."
    SOURCE = Path(selection["source_path"])
    NEW_TASK = selection["new_task"]
    print("Selected model:", SOURCE)
    print("Saved tasks:", selection["completed_tasks"])
    print("Only task", NEW_TASK, "will be taught.")
    print("Beta values:", BETAS)


## 3. Inspect and save your choice on CPU

Replace `None` with the number printed beside the file you want. Run the cell. It prints the actual tasks and scores stored in that file, then saves your selection to Drive.

A valid file has learned 3, 0, and 9, optionally followed by 5, using Hyperfan-in and no forgetting. If you choose another file, the cell stops and explains why. It never silently starts a fresh model.

After seeing the correct saved scores, finish the CPU session using the last cell. Open a fresh A100 session, run cell 1, change MODE to RUN in cell 2, and continue downward. You do not need to select the number again.


In [ ]:
SOURCE_NUMBER = None  # In SELECT mode, replace None with a number from the list.
if MODE == "SELECT":
    assert isinstance(SOURCE_NUMBER, int) and 1 <= SOURCE_NUMBER <= len(candidates), "Set SOURCE_NUMBER to a number from the printed list."
    SOURCE = candidates[SOURCE_NUMBER - 1]
    details = inspect_source(SOURCE)
    completed = details["completed_tasks"]
    print("Saved model:", SOURCE)
    print("Completed tasks:", completed)
    print("Saved validation scores:", details["accuracies"])
    print("Initialization:", details["config"]["initialization"])
    print("Learning rate:", details["config"]["learning_rate"])
    assert details["config"]["initialization"] == "hyperfan_in", "Select a model from the revised-initialization experiment."
    assert details["config"]["dataset"] == "tiny_imagenet" and details["config"]["backbone"] == "resnet50"
    assert completed in (["3", "0", "9"], ["3", "0", "9", "5"]), "Select a checkpoint after learning 9 or 5. The last model after learning 17 is too late."
    NEW_TASK = "17" if completed[-1] == "5" else "5"
    selection = {"source_path": str(SOURCE), "source_sha256": file_hash(SOURCE),
        "completed_tasks": completed, "new_task": NEW_TASK,
        "saved_accuracies": details["accuracies"], "code_version": CODE_VERSION}
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    if SELECTION_PATH.exists():
        assert json.loads(SELECTION_PATH.read_text()) == selection, "This experiment already has another selection. Choose a new experiment name."
    else:
        assert not any(OUTPUT_ROOT.iterdir()), "This folder has other files. Choose a new experiment name."
        SELECTION_PATH.write_text(json.dumps(selection, indent=2) + "\n")
    print("Selection saved. Next lesson:", NEW_TASK)
else:
    print("Using the selection saved during the CPU session.")


## 4. Run the single new lesson on GPU

In SELECT mode this cell does nothing. In RUN mode it loads the selected model, checks its scores against the saved scores, and teaches only the next task.

After every pass through the images, it prints new-task loss, beta times the protection penalty, and all task scores. The report also saves the unweighted penalty and gradient measurements. Losses are averages over training batches; task scores are measured afterwards on validation images.

If you later try several beta values, every value loads the same source file independently. A stronger-beta run never starts from a previous beta run's final model. This tests beta for this one lesson, not beta throughout the entire learning sequence.


In [ ]:
reports = []
if MODE == "RUN":
    from uncle.config import Config
    from uncle.streams import build_tasks
    source_details = inspect_source(SOURCE)
    assert source_details["completed_tasks"] == selection["completed_tasks"]
    assert file_hash(SOURCE) == selection["source_sha256"], "The selected source file has changed. Inspect it again with a new experiment name."
    source_config = Config(**source_details["config"])
    task_order = (*selection["completed_tasks"], NEW_TASK)
    tasks = build_tasks(source_config, root=DATA, include=task_order, download=True)
    def show_epoch(row):
        print(f"Pass {row['epoch']}: new-task loss = {row['task_loss']:.4f}; beta ? protection = {row['weighted_protection_loss']:.4f}")
        print("Validation scores:", " | ".join(f"task {t}: {row['accuracies'][t]:.1f}%" for t in task_order), flush=True)
    for beta in BETAS:
        label = format(beta, ".8g").replace(".", "_")
        output = OUTPUT_ROOT / f"beta_{label}"
        print("Loading the selected source model. Beta:", beta, flush=True)
        report = run_learning_diagnostic(SOURCE, output, NEW_TASK, tasks,
            beta=beta, device="cuda", code_version=CODE_VERSION, on_epoch=show_epoch)
        assert report["source_sha256"] == selection["source_sha256"]
        reports.append((output, report))
        print("Final scores:", report["final_accuracies"])
        print("Saved model:", output / "checkpoint.pt", flush=True)
        torch.cuda.empty_cache()
    comparisons = [summarize_diagnostic(report) for _, report in reports]
    print("Beta | new-task score | mean change in old scores | largest old score drop")
    for row in comparisons:
        print(f"{row['beta']:g} | {row['new_task_accuracy']:.1f}% | {row['mean_old_change']:+.1f} points | {row['largest_old_drop']:.1f} points")
    (OUTPUT_ROOT / "beta_comparison.json").write_text(json.dumps(comparisons, indent=2) + "\n")
    print("All requested lessons finished. Run the final cell to check files and release the GPU.")
else:
    print("SELECT mode: no GPU training. After saving your selection, run the final cell.")


## 5. Check saved files and close the session

Run this cell in either mode. In RUN mode it checks that the saved history has exactly one additional lesson and keeps the source history unchanged. It flushes Drive before releasing the GPU.

Share the five pass-by-pass loss and accuracy rows from the default run. We will use them to decide whether a beta search is justified. The report keeps more detail in `report.json`.


In [ ]:
from google.colab import runtime
if MODE == "RUN":
    from uncle import checkpoint as checkpointing
    assert reports and len(reports) == len(BETAS)
    for output, report in reports:
        saved_report = json.loads((output / "report.json").read_text())
        assert saved_report == report and report["status"] == "complete"
        saved_config = dict(report["config"])
        saved_config["tasks"] = tuple(saved_config["tasks"])
        saved_config["hidden"] = tuple(saved_config["hidden"])
        saved_config["requests"] = tuple(tuple(r) for r in saved_config["requests"])
        final = checkpointing.load(output / "checkpoint.pt", Config(**saved_config))
        assert len(final["history"]) == len(report["source_history"]) + 1
        assert json.loads(json.dumps(final["history"][:-1])) == report["source_history"]
        assert final["history"][-1]["task"] == NEW_TASK
        assert final["previous"] == report["final_accuracies"]
        assert len(report["epochs"]) == report["config"]["epochs"]
        assert not final["forgotten"]
        print("Checked:", output)
        del final
    print("Files checked. Saving Drive and ending the GPU session.", flush=True)
else:
    assert SELECTION_PATH.is_file(), "Save your source selection before closing this CPU session."
    print("Source selection saved. Next, use a fresh A100 session and RUN mode.", flush=True)
drive.flush_and_unmount(timeout_ms=600000)
if MODE == "RUN":
    runtime.unassign()
